# Day 05：核心数据结构（下）——字典与 JSON 重点日（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-005` ｜ **开发分支**：`feat/day05-dict-json-api-parser`  
> **配套 HTML5 课件**：`courseware/day05/index.html` ｜ **独立版本快照**：`versions/day05/`

---

## 🎙️ 老陈开场旁白

今天是第一周含金量最高的一天——字典（dict）与 JSON！我不夸张地讲：在大模型应用开发里，你每天睁眼闭眼打交道的全是嵌套字典和 JSON。OpenAI、DeepSeek、通义千问的请求体是 JSON，返回体 `choices[0]['message']['content']` 还是多层嵌套 JSON！很多新手调 API 老是报 `KeyError` 或 `json.JSONDecodeError`，或者导出中文变成 `\u661f\u8fb0` 乱码。今天 STARMIND-005 咱们直接拿一份真实的 OpenAI/DeepSeek 兼容格式模拟返回 JSON 来解剖，手写安全提取器与 UTF-8 落盘工具，彻底打通 JSON 任督二脉。

### 承上启下
- **昨日基线**：昨天咱们用 list、tuple、set 搞定了待办管理器和知识库标签去重，掌握了序列与集合容器。
- **明日预告**：明天（Day 6）咱们学习函数封装（默认参数、*args/**kwargs、lambda、递归），把前面五天散落的脚本重构成高内聚的函数式架构。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day05'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-005）

星辰智脑即将对接 DeepSeek 与通义千问 API，研发团队需要先构建健壮的 JSON 响应解析与审计落盘层，防止上游字段缺失导致服务崩溃。

**今日迭代目标：**
- 熟练掌握 dict 增删改查、`.get()`、`.items()`、字典推导式与多层嵌套解析。
- 精通 `json.loads`、`json.dumps`、`json.load`、`json.dump` 四大方法及 `ensure_ascii=False`。
- 实现对模拟大模型 API JSON 响应的安全提取、费用计算与本地文件持久化。

**验收标准：**
- 能从标准 OpenAI 兼容 JSON 响应中准确提取回复文本、模型名、结束原因与 Token 用量。
- 面对缺失字段或非法 JSON 字符串时返回兜底错误结构而不抛出未捕获异常。
- 能将解析后的审计记录以可读中文 JSON 写入文件并无损读回，pytest 全绿。

---

## 📚 课堂核心技术笔记

### 1. 1. 字典核心操作与 O(1) 查找

字典存储键值对 `{key: value}`，键必须是不可变可哈希对象（如 `str`、`int`、`tuple`）。Python 3.7+ 保证字典按插入顺序遍历。高频方法：`.get(k, default)` 安全取值、`.items()` 遍历键值、`|` 合并两个字典（Python 3.9+：`merged = base | override`）。

> **⚠️ 老陈避坑**：永远不要在不确定键是否存在时直接 `d['content']`，线上接口少返回一个可选字段就会直接触发 `KeyError` 导致 500 报错！

### 2. 2. JSON 四大函数口诀：带 s 管字符串，不带 s 管文件

- `json.loads(s)`：JSON 字符串 -> Python dict/list
- `json.dumps(obj, ensure_ascii=False, indent=2)`：Python 对象 -> 美化后的中文 JSON 字符串
- `json.load(fp)`：从打开的文件对象读取并解析为 Python 对象
- `json.dump(obj, fp, ensure_ascii=False, indent=2)`：将 Python 对象写入文件

> **⚠️ 老陈避坑**：JSON 标准规定：1）键和字符串必须用**双引号**（单引号非法）；2）末尾不能有多余逗号；3）布尔值小写 `true/false`，空值是 `null`；4）JSON 不支持 `set` 类型，序列化前要先转 `list`。

### 3. 3. 真实大模型 Chat Completions 响应解剖

国内主流模型（DeepSeek、通义千问 DashScope 兼容模式、智谱 GLM）均遵循 OpenAI 响应格式：`{'id': '...', 'model': 'deepseek-chat', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '...'}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 20, 'completion_tokens': 50, 'total_tokens': 70}}`。

> **⚠️ 老陈避坑**：注意 `choices` 是一个列表（即使通常只有 1 项），取回复正文要先判断 `choices` 非空，再取 `choices[0]`。


## 💻 实操步骤 1：步骤一：编写嵌套字典安全路径提取与配置合并工具

- **目标文件**：`versions/day05/src/day05_dict_utils.py`
- **老陈代码旁白**：先造一把趁手的瑞士军刀 `safe_get_path`，以后面对 5 层嵌套的大模型 JSON，一行代码安全取到底，再也不怕 KeyError 和 IndexError。


In [ ]:
"""
STARMIND-005 模块一：嵌套字典安全提取与配置合并工具。
演示 dict 操作、isinstance 类型判断、嵌套 list/dict 穿透与 Python 3.9+ 字典合并运算符 |。
"""

from typing import Any, Dict


def safe_get_path(data: Any, path: str, default: Any = None) -> Any:
    """
    通过点分隔路径安全访问多层嵌套的 dict 或 list。
    示例：safe_get_path(resp, "choices.0.message.content", "")
    """
    current = data
    for segment in path.split("."):
        if isinstance(current, dict):
            if segment in current:
                current = current[segment]
            else:
                return default
        elif isinstance(current, list):
            # 列表路径段必须是合法非负整数索引
            if segment.isdigit():
                idx = int(segment)
                if 0 <= idx < len(current):
                    current = current[idx]
                else:
                    return default
            else:
                return default
        else:
            return default
    return current


def merge_model_config(base_cfg: Dict[str, Any], user_override: Dict[str, Any]) -> Dict[str, Any]:
    """合并默认模型参数与用户覆盖参数，不修改原始字典，并过滤掉 None 值。"""
    # 使用字典推导式剔除用户传入的 None，避免覆盖掉有效默认值
    valid_override = {k: v for k, v in user_override.items() if v is not None}
    # Python 3.9+ 字典合并运算符 | 返回全新字典
    return base_cfg | valid_override


if __name__ == "__main__":
    sample = {"choices": [{"message": {"role": "assistant", "content": "你好，我是星辰智脑！"}}]}
    print("提取正文:", safe_get_path(sample, "choices.0.message.content", "默认回复"))
    print("缺失路径:", safe_get_path(sample, "choices.1.message.content", "未命中"))

    default_cfg = {"model": "deepseek-chat", "temperature": 0.7, "max_tokens": 1024}
    final_cfg = merge_model_config(default_cfg, {"temperature": 0.2, "max_tokens": None})
    print("合并配置:", final_cfg)


### 🔍 步骤 1 代码深度解析

1. `safe_get_path` 同时兼容 `dict` 键查找和 `list` 数字索引，彻底告别冗长的 `if 'choices' in d and len(d['choices']) > 0 ...`。
2. `base_cfg | valid_override` 生成新字典，不会意外篡改全局默认配置 `default_cfg`。

**预期运行输出参考：**
```text
提取正文: 你好，我是星辰智脑！
缺失路径: 未命中
合并配置: {'model': 'deepseek-chat', 'temperature': 0.2, 'max_tokens': 1024}
```


## 💻 实操步骤 2：步骤二：实现大模型 API JSON 响应解析器与本地审计落盘

- **目标文件**：`versions/day05/src/day05_json_api_parser.py`
- **老陈代码旁白**：现在拿一份标准 OpenAI/DeepSeek 兼容的模拟 JSON 响应，演示 `json.loads / dumps / load / dump` 全流程，并计算单次调用成本。


In [ ]:
"""
STARMIND-005 模块二：模拟大模型 API JSON 响应解析与审计落盘工具。
演示 json.loads、json.dumps、json.load、json.dump 及 UTF-8 中文无转义持久化。
"""

import json
from pathlib import Path
from typing import Dict, Any

from day05_dict_utils import safe_get_path

# 模拟一条真实的 DeepSeek / 通义千问兼容格式 API JSON 响应
MOCK_API_RESPONSE_JSON = json.dumps(
    {
        "id": "chatcmpl-starmind-005",
        "object": "chat.completion",
        "model": "deepseek-chat",
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": "您好！您的订单 ORD-20260505 已由顺丰揽收，预计明日送达杭州滨江区。",
                },
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": 128,
            "completion_tokens": 42,
            "total_tokens": 170,
        },
    },
    ensure_ascii=False,
)


def parse_llm_response(raw_json: str) -> Dict[str, Any]:
    """解析大模型原始 JSON 响应字符串，返回星辰智脑内部标准结果字典。"""
    try:
        payload = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        return {
            "ok": False,
            "error": f"JSON 解析失败: {exc.msg}",
            "reply": "",
            "total_tokens": 0,
        }

    # 如果上游返回了错误对象（如限流或鉴权失败）
    if isinstance(payload, dict) and "error" in payload:
        return {
            "ok": False,
            "error": str(safe_get_path(payload, "error.message", payload["error"])),
            "reply": "",
            "total_tokens": 0,
        }

    reply_text = safe_get_path(payload, "choices.0.message.content", "")
    model_name = safe_get_path(payload, "model", "unknown-model")
    finish_reason = safe_get_path(payload, "choices.0.finish_reason", "unknown")
    prompt_tokens = int(safe_get_path(payload, "usage.prompt_tokens", 0))
    completion_tokens = int(safe_get_path(payload, "usage.completion_tokens", 0))
    total_tokens = int(safe_get_path(payload, "usage.total_tokens", prompt_tokens + completion_tokens))

    return {
        "ok": bool(reply_text),
        "request_id": safe_get_path(payload, "id", "N/A"),
        "model": model_name,
        "reply": reply_text,
        "finish_reason": finish_reason,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
    }


def save_audit_record(record: Dict[str, Any], file_path: Path) -> Path:
    """将结构化解析结果以可读中文 JSON 写入磁盘文件。"""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with file_path.open("w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    return file_path


def load_audit_record(file_path: Path) -> Dict[str, Any]:
    """从磁盘 JSON 文件无损读回审计记录。"""
    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


if __name__ == "__main__":
    parsed = parse_llm_response(MOCK_API_RESPONSE_JSON)
    # 使用 json.dumps 格式化打印，ensure_ascii=False 确保中文直接显示
    print(json.dumps(parsed, ensure_ascii=False, indent=2))


### 🔍 步骤 2 代码深度解析

1. `parse_llm_response` 同时处理了三种真实生产场景：合法响应、上游返回 `{"error": ...}` 业务错、以及网关超时截断导致的非法 JSON。
2. `file_path.parent.mkdir(parents=True, exist_ok=True)` 确保父目录不存在时自动创建，避免 `FileNotFoundError`。

**预期运行输出参考：**
```text
{
  "ok": true,
  "request_id": "chatcmpl-starmind-005",
  "model": "deepseek-chat",
  "reply": "您好！您的订单 ORD-20260505 已由顺丰揽收，预计明日送达杭州滨江区。",
  "finish_reason": "stop",
  "prompt_tokens": 128,
  "completion_tokens": 42,
  "total_tokens": 170
}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day05/tests/test_day05.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-005 自动化单元测试。"""

import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day05_dict_utils import safe_get_path, merge_model_config
from day05_json_api_parser import (
    MOCK_API_RESPONSE_JSON,
    parse_llm_response,
    save_audit_record,
    load_audit_record,
)


def test_safe_get_path_and_merge():
    data = {"a": [{"b": "ok"}]}
    assert safe_get_path(data, "a.0.b") == "ok"
    assert safe_get_path(data, "a.5.b", "fallback") == "fallback"
    assert safe_get_path(data, "x.y", None) is None

    merged = merge_model_config({"temp": 0.7, "top_p": 0.9}, {"temp": 0.1, "top_p": None})
    assert merged == {"temp": 0.1, "top_p": 0.9}


def test_parse_llm_response_variants():
    ok_res = parse_llm_response(MOCK_API_RESPONSE_JSON)
    assert ok_res["ok"] is True
    assert ok_res["total_tokens"] == 170
    assert "顺丰" in ok_res["reply"]

    bad_res = parse_llm_response("{not valid json")
    assert bad_res["ok"] is False
    assert "JSON 解析失败" in bad_res["error"]

    err_res = parse_llm_response(json.dumps({"error": {"message": "rate limit exceeded"}}))
    assert err_res["ok"] is False
    assert "rate limit" in err_res["error"]


def test_json_save_and_load_roundtrip(tmp_path: Path):
    record = parse_llm_response(MOCK_API_RESPONSE_JSON)
    target = tmp_path / "logs" / "audit.json"
    save_audit_record(record, target)
    raw_text = target.read_text(encoding="utf-8")
    assert "顺丰" in raw_text  # 确保未转义为 \\uXXXX
    reloaded = load_audit_record(target)
    assert reloaded == record

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：词频统计字典构建器

**题目要求：**
给定一组客服工单意图标签列表 `['物流', '退款', '物流', '发票', '物流', '退款']`，编写 `count_intents(labels: list[str]) -> dict[str, int]`，统计每个意图出现次数并按频次从高到低排序返回字典。

**思路提示：** 用 `counts[label] = counts.get(label, 0) + 1` 累计，再用 `sorted(counts.items(), key=lambda x: x[1], reverse=True)` 排序。


In [ ]:
"""Day 5 课后题 1：工单意图频次统计器。"""

from typing import List, Dict


def count_intents(labels: List[str]) -> Dict[str, int]:
    """统计每个意图标签出现次数，并按频次降序返回字典。"""
    counts: Dict[str, int] = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    # 按值降序排列，Python 3.7+ 字典保留插入顺序
    sorted_pairs = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
    return dict(sorted_pairs)


if __name__ == "__main__":
    sample = ["物流", "退款", "物流", "发票", "物流", "退款"]
    print("意图统计:", count_intents(sample))


**答案解析与评分标准：**

考查 `dict.get(k, 0) + 1` 经典计数范式与按 value 排序技巧。


### 作业 2（企业实战）：企业实战：多模型账单按部门分组汇总

**题目要求：**
输入一组 API 调用记录列表（每条含 `dept` 和 `tokens`），编写 `aggregate_by_dept(records: list[dict]) -> dict[str, int]`，汇总各部门总 Token 消耗并输出美化 JSON。

**思路提示：** 遍历列表，使用 `totals[dept] = totals.get(dept, 0) + tokens`。


In [ ]:
"""Day 5 课后题 2：按部门汇总大模型 Token 消耗账单。"""

import json
from typing import List, Dict, Any


def aggregate_by_dept(records: List[Dict[str, Any]]) -> Dict[str, int]:
    """按部门分组累加 Token 用量。"""
    totals: Dict[str, int] = {}
    for rec in records:
        dept = str(rec.get("dept", "未知部门"))
        tokens = int(rec.get("tokens", 0))
        totals[dept] = totals.get(dept, 0) + tokens
    return totals


if __name__ == "__main__":
    logs = [
        {"dept": "客服部", "tokens": 1200},
        {"dept": "研发部", "tokens": 3500},
        {"dept": "客服部", "tokens": 800},
    ]
    print(json.dumps(aggregate_by_dept(logs), ensure_ascii=False, indent=2))


**答案解析与评分标准：**

企业计费与成本分摊（FinOps）的基础聚合模型，注意缺失字段兜底默认值。


### 作业 3（架构进阶）：架构进阶：JSON Schema 必填字段校验器

**题目要求：**
大模型输出结构化 JSON 时偶尔会漏掉关键字段。编写 `validate_payload(payload: dict, required_types: dict[str, type]) -> tuple[bool, list[str]]`，检查字典是否包含所有必填键且值类型符合预期，返回 `(是否通过, 错误明细列表)`。

**思路提示：** 遍历 `required_types.items()`，先查 `key not in payload`，再查 `not isinstance(payload[key], expected_type)`。


In [ ]:
"""Day 5 课后题 3：轻量级 JSON 响应结构校验器。"""

from typing import Dict, Any, Tuple, List, Type


def validate_payload(
    payload: Dict[str, Any], required_types: Dict[str, Type[Any]]
) -> Tuple[bool, List[str]]:
    """校验字典是否包含全部必填字段且满足类型要求。"""
    errors: List[str] = []
    for field, expected_type in required_types.items():
        if field not in payload:
            errors.append(f"缺少必填字段: {field}")
        elif not isinstance(payload[field], expected_type):
            actual = type(payload[field]).__name__
            errors.append(f"字段 {field} 类型错误: 期望 {expected_type.__name__}, 实际 {actual}")
    return (len(errors) == 0, errors)


if __name__ == "__main__":
    schema = {"intent": str, "confidence": float, "need_human": bool}
    print("合法:", validate_payload({"intent": "退款", "confidence": 0.95, "need_human": False}, schema))
    print("异常:", validate_payload({"intent": "退款", "confidence": "高"}, schema))


**答案解析与评分标准：**

这就是 Day 23 Pydantic 数据校验库最核心的底层设计思想：把运行时弱类型字典转化为强契约校验。


---

## 🎯 老陈收尾复盘

今天把嵌套 dict 和 `json.loads / dumps / load / dump` 摸透后，你以后看到任何第三方大模型 API 文档都不会发怵了。临走前再默念一遍两个保命参数：`dict.get(key, default)` 防 KeyError，`json.dumps(..., ensure_ascii=False, indent=2)` 防中文变 Unicode 转义码！
